In [5]:
import baostock as bs
import pandas as pd
from pathlib import Path

# 增加了 adjust 参数，默认是 "hfq"
def fetch_daily_baostock(symbol: str, start: str, end: str, adjust: str = "hfq"):
    start_fmt = f"{start[:4]}-{start[4:6]}-{start[6:]}"
    end_fmt = f"{end[:4]}-{end[4:6]}-{end[6:]}"
    
    if symbol.startswith('6'):
        bs_code = f"sh.{symbol}"
    else:
        bs_code = f"sz.{symbol}"
    
    # 增加了一个字典，把 hfq/qfq 翻译成 baostock 认识的 1/2/3
    adj_map = {"hfq": "1", "qfq": "2", "": "3"}
    adj_flag = adj_map.get(adjust, "1") # 默认用1
        
    rs = bs.query_history_k_data_plus(bs_code,
        "date,code,open,high,low,close,volume,amount",
        start_date=start_fmt, end_date=end_fmt,
        frequency="d", adjustflag=adj_flag) # 这里传入动态的 adj_flag
    
    data_list = []
    while (rs.error_code == '0') & rs.next():
        data_list.append(rs.get_row_data())
    
    return pd.DataFrame(data_list, columns=rs.fields)

In [2]:
for s in ["600519", "000001", "601318"]:
    temp_df = pd.read_csv(f"data/raw/{s}.csv")
    print(f"\n=== {s} 最新3天收盘价 ===")
    print(temp_df[['date', 'close']].tail(3))


=== 600519 最新3天收盘价 ===
            date         close
1661  2026-11-12  10815.723069
1662  2026-11-13  11007.377802
1663  2026-11-16  11150.332752

=== 000001 最新3天收盘价 ===
            date        close
1661  2026-11-12  1505.145450
1662  2026-11-13  1540.891058
1663  2026-11-16  1570.253522

=== 601318 最新3天收盘价 ===
            date       close
1661  2026-11-12  175.744642
1662  2026-11-13  177.293488
1663  2026-11-16  180.094594
